In [ ]:
import pandas as pd
import numpy as np

# 剧毒原始数据：融合了 DataCamp 全章节的脏乱差
df_users = pd.DataFrame({
    'user_id': ['U01', 'U 002', 'U@03', 'U  01', 'U0400000', 'UA134', 'U3786A'], # 注意：有重复数据
    'birth_year': ['1990', '1985', '2050', '1990', '1992', np.nan, '1988'], # 注意：有未来年份、缺失值
    'age': [36, 41, -5, 36, 33, 25, 38], # 假设当前是 2026 年
    'income': ['$5,000', '6000.50', 'NaN', '$5,000', '-1000', ' 8000 ', '$$9000'], # 字符串混杂符号、非法负数
    'company': ['Tencent', ' Alibaba ', 'ByteDance', 'Tencent', 'Alibba', 'McDonalds', 'Apple Inc']
})

print("🚨 原始剧毒数据集：")
print(df_users.info())

In [ ]:
# back up raw data
df_clean = df_users.copy()

In [ ]:
# cleaning user_id

# anomalous-length boolean sequence and raw data 
is_anomalous_length = (df_clean['user_id'].str.len() < 2) | (df_clean['user_id'].str.len() > 6)
anomalous_length_data = df_clean[is_anomalous_length]
normal_length_data = df_clean[~is_anomalous_length]
print(anomalous_length_data['user_id'].str.len().value_counts().head())

# define data standard patterns via regex
standard_pattern_id = r'^U\d{2,5}$'
is_nonstandard_pattern_id = ~normal_length_data['user_id'].str.match(standard_pattern_id,na=False)
nonstandard_pattern_id = normal_length_data[is_nonstandard_pattern_id]
print(nonstandard_pattern_id.head())


# ==========================================
# 动作 1：纯粹的清洗引擎 (只管洗，绝对不删减行数！)
# ==========================================
def clean_id_column(raw_id_series):
    # 暴力提取纯数字并加上 U
    pure_digits = raw_id_series.str.replace(r'\D', '', regex=True)
    cleaned_id = 'U' + pure_digits
    
    # 终极掩码与分流
    standard_pattern = r'^U\d{2,5}$'
    valid_mask = cleaned_id.str.match(standard_pattern, na=False)
    
    # 进多少行，出多少行，不多不少
    return np.where(valid_mask, cleaned_id, 'Unknown')

# ==========================================
# 动作 2：表级流水线控制 (在这里删减、去重)
# ==========================================
# 1. 执行原位清洗 (长度绝对一致，完美塞入)
df_clean['user_id'] = clean_id_column(df_clean['user_id'])

# 2. 拿到全表之后，开始分离废料
df_unknown = df_clean[df_clean['user_id'] == 'Unknown']
df_valid = df_clean[df_clean['user_id'] != 'Unknown']

# 3. 对合法全表进行去重
df_valid = df_valid.drop_duplicates(subset=['user_id'], keep='last')

# 4. 合流
df_final = pd.concat([df_valid, df_unknown])

print(df_final)

    




In [ ]:
import numpy as np
import pandas as pd

dirty_data = [
    "1990",                # 原始年份 (OK)
    "1985-05-12",          # ISO格式 (OK)
    "2050/13/01",          # 逻辑错误：月份溢出
    "1990.0",              # 浮点型转字符串残留 (Common in Excel)
    "92-03-15",            # 两位数年份缩写 (歧义：1992 还是 2092?)
    np.nan,                # 缺失值
    "1988/08/08 00:00:00", # 带时间戳的冗余格式
    " 1991 ",              # 首尾空格
    "1990年6月",            # 中文环境混合
    "05/01/1993",          # 美式 vs 英式歧义 (MM/DD 还是 DD/MM?)
    "Unknown",             # 垃圾占位符
    "1989.12.12",          # 点号分隔符
    "19-02-1987",          # 顺序颠倒 (DD-MM-YYYY)
    "19941020"             # 紧凑型字符串
]

df_time = pd.DataFrame({'birth_date': dirty_data})
print(df_time)

In [ ]:
print(df_time['birth_date'].value_counts())

In [ ]:
import pandas as pd
import numpy as np
import re

def clean_birth_date_pipeline(df):
    # 1. 备份原始数据（这是专业素养：保证可溯源）
    df['raw_input'] = df['birth_date']
    
    # 2. 预处理：去空格、统一转字符串
    df['birth_date'] = df['birth_date'].astype(str).str.strip().replace('nan', np.nan)
    
    # 3. 第一轮：尝试标准解析
    df['parsed_date'] = pd.to_datetime(df['birth_date'], errors='coerce', format='mixed')
    
    # 4. 第二轮：针对失败者（mask）进行正则抢救
    mask = df['birth_date'].notna() & df['parsed_date'].isna()
    
    def heuristic_rescue(val):
        if not isinstance(val, str): return None
        # 寻找所有连续的4位数字（假设那是年份）
        years = re.findall(r'\d{4}', val)
        if years:
            # 如果年份大于当前年份，记录为逻辑错误
            year = int(years[0])
            if 1900 <= year <= 2026:
                return pd.Timestamp(year=year, month=1, day=1)
        return None

    df.loc[mask, 'parsed_date'] = df.loc[mask, 'birth_date'].apply(heuristic_rescue)
    
    return df

In [ ]:
import pandas as pd

# 极度混乱的注册时间数据
df_time = pd.DataFrame({
    'user_id': ['U101', 'U102', 'U103', 'U104', 'U105', 'U106'],
    'reg_date': [
        '2026-05-11',       # 陷阱1：标准格式，但它是字符串
        '11/05/2026',       # 陷阱2：英式/欧式格式 (日/月/年)
        '2026年05月12日',     # 陷阱3：中文夹杂
        'Not a Date',       # 陷阱4：纯垃圾文本
        '2999-12-31',       # 陷阱5：时空穿越者 (未来的时间)
        '1990-01-01'        # 陷阱6：远古时空 (公司2010年才成立)
    ]
})

print("🚨 原始时间数据：")
print(df_time)

In [ ]:
import pandas as pd
import numpy as np
import re

# back up raw data
df_time['raw_input'] = df_time['reg_date']

# preprocessing:remove spaces,uniformly convert to string
df_time['raw_input'] = df_time['raw_input'].astype(str).str.strip().replace('nan',np.nan,regex=False)

# round one:attempt standard analysis
df_time['parsed_date'] = pd.to_datetime(df_time['raw_input'],format='mixed',errors='coerce')

# round two:perform regular expression rescue on the losers(mask)
mask = df_time['raw_input'].notna() & df_time['parsed_date'].isna()
rescued_date = (
    df_time.loc[mask,'raw_input']
    .str.replace(r'\D','-',regex=True)
    .str.strip('-')
    .pipe(pd.to_datetime,errors='coerce')
)

df_time['parsed_date'] = df_time['parsed_date'].fillna(rescued_date)

is_valid_date = df_time['parsed_date'].between('2010-01-01', '2026-05-11')

df_time['final_parsed_date'] = df_time['parsed_date'].where(is_valid_date, pd.NaT)

df_time = df_time.drop(columns=['reg_date','raw_input','parsed_date']).rename(columns={'final_parsed_date':'reg_date'})
print(df_time)


In [51]:
import pandas as pd
import numpy as np

# 极度恶劣的金融交易时间数据
df_trade = pd.DataFrame({
    'trade_id': ['T01', 'T02', 'T03', 'T04', 'T05', 'T06'],
    'trade_time': [
        '2026-05-11 14:30:00', # 正常数据
        '11-May-2026',         # 陷阱1：英文月份缩写格式
        'Pending',             # 陷阱2：业务代号（未成交）混入了时间列
        '2026/05/15 25:00:00', # 陷阱3：绝对的非法时间 (一天哪有25点？)
        '2025-12-31',          # 陷阱4：去年的历史数据
        ''                     # 陷阱5：隐形的空字符串
    ]
})

print("🚨 原始交易数据：")
print(df_trade)

🚨 原始交易数据：
  trade_id           trade_time
0      T01  2026-05-11 14:30:00
1      T02          11-May-2026
2      T03              Pending
3      T04  2026/05/15 25:00:00
4      T05           2025-12-31
5      T06                     


In [52]:
import pandas as pd
import numpy as np

# back to raw data
df_trade['raw_input'] = df_trade['trade_time']

# preprocessing:remove spaces,unifomly convert to string
df_trade['raw_input'] = df_trade['raw_input'].astype(str).str.strip().replace('nan',np.nan)

# round one:attempt standard analysis
df_trade['parsed_date'] = pd.to_datetime(df_trade['raw_input'],format='mixed',errors='coerce')


is_valid_date = df_trade['parsed_date'].dt.year >= 2026
df_trade['final_date'] = df_trade['parsed_date'].where(is_valid_date,pd.NaT)


df_trade=df_trade.drop(columns=['trade_time','raw_input','parsed_date']).rename(columns={'final_date':'trade_time'})
print(df_trade)

  trade_id          trade_time
0      T01 2026-05-11 14:30:00
1      T02 2026-05-11 00:00:00
2      T03                 NaT
3      T04                 NaT
4      T05                 NaT
5      T06                 NaT
